# Experiment 2.1b-i: Equivariance Error vs Depth

## Paired notebook for `run_experiment.py`

This notebook is the analysis/reporting counterpart of the script in the same directory:

- `run_experiment.py` contains the executable experiment kernel.
- This notebook imports that script, runs the same experiment through its `run_experiment()`
  interface, and then presents the returned results.

That design is deliberate: the notebook no longer duplicates the core training loop, so the
script and notebook stay aligned by construction.


---
## Exact Question, Metric, and Scope

### Toy-study question

For a **deep linear network** trained with the Muon update rule, how does the final
**target-layer equivariance error** depend on the position of the layer that was bilaterally
conjugated at initialization and on the total network depth?

### Implemented metric

For a chosen target layer $\ell$:

$$
\varepsilon_\ell = \frac{\|W^{(B)}_\ell - R\,W^{(A)}_\ell\,S^\top\|_F}{\|W^{(A)}_\ell\|_F}
$$

where:

- **Path A:** train from the original initialization,
- **Path B:** conjugate **only** layer $\ell$ at initialization, then retrain on the same data,
- $R, S \in O(d)$ are random orthogonal matrices,
- the score is measured **only at the same target layer** after training.

### What this notebook does and does not claim

This notebook **does** characterize the implemented toy metric above as a function of depth and
layer position.

This notebook **does not** by itself provide:

- a proof of the mechanism behind any depth dependence,
- a full cross-layer propagation measurement,
- a causal demonstration of correlation length,
- a general statement beyond this linear-network / fixed-hyperparameter toy setup.


In [ ]:
import importlib.util
import os
import platform
import sys
import time
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = None

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

SCRIPT_RELATIVE_PATH = Path(
    'experiments/Tier_2_Symmetry_Reparametrization_Tests/'
    'Exp_2.1_Conjugation_Covariance/'
    '2.1b-i_Equivariance_Error_vs_Depth/'
    'run_experiment.py'
)


def find_repo_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / SCRIPT_RELATIVE_PATH).exists():
            return candidate
    raise FileNotFoundError(
        f'Could not locate repo root from {start}; expected to find {SCRIPT_RELATIVE_PATH}.'
    )


def ascii_table(headers, rows):
    headers = [str(h) for h in headers]
    rows = [[str(v) for v in row] for row in rows]
    widths = [
        max(len(headers[i]), max((len(row[i]) for row in rows), default=0))
        for i in range(len(headers))
    ]

    def _format(row):
        return ' | '.join(row[i].ljust(widths[i]) for i in range(len(widths)))

    separator = '-+-'.join('-' * width for width in widths)
    return '\n'.join([_format(headers), separator, *[_format(row) for row in rows]])


def all_edge_higher_than_center(depth_results):
    return all(depth_results[d]['edge_mean_error'] > depth_results[d]['center_mean_error'] for d in depth_results)


def all_negative_mean_profile_correlations(depth_results):
    return all(depth_results[d]['mean_profile_correlation'] < 0.0 for d in depth_results)


REPO_ROOT = find_repo_root()
SCRIPT_PATH = REPO_ROOT / SCRIPT_RELATIVE_PATH
EXPERIMENT_DIR = SCRIPT_PATH.parent

spec = importlib.util.spec_from_file_location('exp_2_1b_i_equivariance_depth', SCRIPT_PATH)
if spec is None or spec.loader is None:
    raise ImportError(f'Unable to create import spec for {SCRIPT_PATH}')
exp = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = exp
spec.loader.exec_module(exp)

SMOKE_MODE = os.environ.get('EQUIV_DEPTH_NOTEBOOK_SMOKE', '0') == '1'
DEFAULT_CONFIG = dict(exp.get_default_config())
ACTIVE_CONFIG = dict(DEFAULT_CONFIG)
RUN_KWARGS = {'verbose': False}
if SMOKE_MODE:
    ACTIVE_CONFIG.update({'depths': [4], 'num_steps': 5, 'num_seeds': 2})
    RUN_KWARGS.update({'depths': [4], 'num_steps': 5, 'num_seeds': 2})


def training_call_count(cfg):
    return sum((1 + depth) * cfg['num_seeds'] for depth in cfg['depths'])


print(f'Repo root:      {REPO_ROOT}')
print(f'Experiment dir: {EXPERIMENT_DIR}')
print(f'Script path:    {SCRIPT_PATH}')
print(f'Python:         {platform.python_version()}')
print(f'NumPy:          {np.__version__}')
print(f'Matplotlib:     {matplotlib.__version__}')
print(f'Smoke mode:     {SMOKE_MODE}')


def show_figure(fig):
    if display is not None:
        display(fig)
    else:
        fig.canvas.draw()


---
## Reproducibility and Runtime Configuration

The default notebook path runs the **same default experiment configuration** as the script.
For automated smoke checks, the notebook also accepts an environment override:

- `EQUIV_DEPTH_NOTEBOOK_SMOKE=1` → use a much smaller config for quick execution.

Interpret the smoke configuration only as a code-path check. The substantive toy-study readout is
intended for the default configuration.


In [ ]:
config_rows = [
    ['dim', ACTIVE_CONFIG['dim']],
    ['depths', ACTIVE_CONFIG['depths']],
    ['num_steps', ACTIVE_CONFIG['num_steps']],
    ['num_seeds', ACTIVE_CONFIG['num_seeds']],
    ['batch_size', ACTIVE_CONFIG['batch_size']],
    ['lr', ACTIVE_CONFIG['lr']],
    ['momentum', ACTIVE_CONFIG['momentum']],
    ['ns_iters', ACTIVE_CONFIG['ns_iters']],
    ['data_seed_rule', f"{ACTIVE_CONFIG['data_seed_base']} + seed * {ACTIVE_CONFIG['data_seed_stride']}"],
    ['weight_seed_rule', f"{ACTIVE_CONFIG['weight_seed_base']} + seed"],
    ['training_runs', training_call_count(ACTIVE_CONFIG)],
    ['optimizer_steps', training_call_count(ACTIVE_CONFIG) * ACTIVE_CONFIG['num_steps']],
]

print('Active configuration')
print(ascii_table(['field', 'value'], config_rows))
print('\nNotebook note: default execution re-runs the full toy experiment and can take about a minute on this machine.')
if SMOKE_MODE:
    print('Smoke-mode note: reduced settings are being used only to verify notebook/script execution paths.')


---
## Primitive Sanity Check: What Is Actually Safe to Claim About Newton--Schulz Here?

The experiment relies on the implemented Newton--Schulz map because it is a matrix-polynomial
update transform with a strong **conjugation-equivariance** property under orthogonal bilateral
transforms. That property is the relevant one here.

By contrast, this notebook avoids claiming that the current `NS_ITERS = 5` implementation always
returns an almost-perfect orthogonal polar factor for arbitrary random matrices. We check both
quantities explicitly below.


In [ ]:
rng = np.random.RandomState(42)
dim = ACTIVE_CONFIG['dim']
M = rng.randn(dim, dim)
R = exp.random_orthogonal(dim, rng)
S = exp.random_orthogonal(dim, rng)

ns_M = exp.newton_schulz(M)
ns_RMS = exp.newton_schulz(R @ M @ S.T)
ns_equiv_err = np.linalg.norm(ns_RMS - R @ ns_M @ S.T) / max(np.linalg.norm(ns_M), 1e-30)
ns_ortho_err = np.linalg.norm(ns_M.T @ ns_M - np.eye(dim), ord='fro')
R_ortho_err = np.linalg.norm(R.T @ R - np.eye(dim), ord='fro')
S_ortho_err = np.linalg.norm(S.T @ S - np.eye(dim), ord='fro')

print('Primitive sanity check')
print(ascii_table(
    ['quantity', 'value'],
    [
        ['relative NS conjugation-equivariance error', f'{ns_equiv_err:.3e}'],
        ['NS orthogonality residual ||Q^T Q - I||_F', f'{ns_ortho_err:.3e}'],
        ['orthogonality residual of sampled R', f'{R_ortho_err:.3e}'],
        ['orthogonality residual of sampled S', f'{S_ortho_err:.3e}'],
    ]
))
print('\nInterpretation: the implementation is numerically conjugation-equivariant in the relevant sense;')
print('the orthogonality residual should be treated as an empirical diagnostic, not silently idealized away.')


---
## Run the Scripted Experiment Kernel

The next cell calls `run_experiment()` from the paired script. This is the only place where the
main sweep is executed. All later analysis cells consume the returned structured results.


In [ ]:
t0 = time.perf_counter()
results = exp.run_experiment(**RUN_KWARGS)
notebook_wall_clock = time.perf_counter() - t0

depth_results = results['depth_results']
used_config = results['config']
depths = used_config['depths']

print('Run completed.')
print(f"Script-reported runtime: {results['runtime_seconds']:.2f} s")
print(f"Notebook wall-clock:     {notebook_wall_clock:.2f} s")
print(f"Metric:                 {results['metric_formula']}")
print(f"Scope:                  {results['scope']}")
print('\nConfig actually used')
print(ascii_table(['field', 'value'], [[k, used_config[k]] for k in used_config]))
print('\nSeed schedule preview (first 3 rows)')
seed_rows = [
    [row['seed_index'], row['data_seed'], row['weight_seed']]
    for row in results['seed_schedule'][: min(3, len(results['seed_schedule']))]
]
print(ascii_table(['seed_index', 'data_seed', 'weight_seed'], seed_rows))
if SMOKE_MODE:
    print('\nSmoke-mode reminder: the following figures/tables verify that the notebook pipeline works,')
    print('but they are not meant to substitute for the default full toy-study run.')


---
## Result 1: Layerwise Equivariance-Error Profiles by Depth

Each panel below shows the final target-layer equivariance error as a function of the target
layer index. The line is the layerwise mean across seeds, the shaded region is a normal-approximate
95% confidence band for that mean, and the faint points are the individual seed-level values.


In [ ]:
fig, axes = plt.subplots(1, len(depths), figsize=(5 * len(depths), 4.2), sharey=True, squeeze=False)
axes = axes.ravel()

for ax, depth in zip(axes, depths):
    r = depth_results[depth]
    errors = np.asarray(r['layer_errors'])
    x = np.arange(depth)
    means = np.asarray(r['mean_errors'])
    ci = np.array([exp.ci95_half_width(errors[:, i]) for i in range(depth)])

    color = f'C{depths.index(depth)}'
    for seed_idx in range(errors.shape[0]):
        ax.scatter(x, errors[seed_idx], color=color, alpha=0.22, s=18)
    ax.plot(x, means, color=color, marker='o', linewidth=2.2, label='mean profile')
    ax.fill_between(x, means - ci, means + ci, color=color, alpha=0.18, label='95% CI')

    ax.set_title(f'Depth {depth}')
    ax.set_xlabel('target layer index')
    ax.set_xticks(x)
    ax.set_ylim(bottom=0.0)
    ax.text(
        0.03,
        0.97,
        f"corr(mean profile) = {r['mean_profile_correlation']:.3f}",
        transform=ax.transAxes,
        ha='left',
        va='top',
        fontsize=9,
        bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.75},
    )

axes[0].set_ylabel('final target-layer equivariance error')
fig.suptitle('Equivariance-error profiles returned by run_experiment()', y=1.02)
fig.tight_layout()
show_figure(fig)
plt.close(fig)


In [ ]:
profile_rows = []
for depth in depths:
    r = depth_results[depth]
    profile_rows.append([
        depth,
        r['center_layers'],
        r['min_error_layer'],
        r['max_error_layer'],
        f"{r['edge_mean_error']:.4f}",
        f"{r['center_mean_error']:.4f}",
        f"{r['edge_minus_center_mean']:.4f} ± {r['edge_minus_center_ci95']:.4f}",
        f"{r['mean_profile_correlation']:.3f}",
    ])

print('Profile summary')
print(ascii_table(
    ['depth', 'center layers', 'min-error layer', 'max-error layer', 'edge mean', 'center mean', 'edge-center', 'corr(mean profile)'],
    profile_rows,
))

if SMOKE_MODE:
    print('\nInterpretation note: smoke mode is only a path check, so do not over-read the qualitative pattern.')
elif all_edge_higher_than_center(depth_results) and all_negative_mean_profile_correlations(depth_results):
    print('\nInterpretation: in the default toy setup, the observed pattern is edge-high / center-low rather than middle-peaked.')
else:
    print('\nInterpretation: inspect the table/figure directly; this run does not show a single clean pattern across all tested depths.')
print('Caveat: this remains a self-error measurement at the conjugated layer, not a full all-layer response map.')


---
## Result 2: Boundary-Distance Summaries and Seed-Level Diagnostics

The script returns enough raw structure to examine boundary-distance summaries more seriously.
Below we look at:

1. **Edge vs center** summaries by seed, where "center" means the middle layer for odd depth or
   the average of the two central layers for even depth.
2. **Per-seed correlations** between `dist_from_edge` and the layerwise error profile.

These are still summaries of the implemented target-layer metric, but they are more informative
than a single correlation computed only on the mean profile.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.2))

# Left panel: edge vs center by seed
ax = axes[0]
positions = []
labels = []
for idx, depth in enumerate(depths):
    r = depth_results[depth]
    edge_x = 2 * idx
    center_x = 2 * idx + 1
    positions.extend([edge_x, center_x])
    labels.extend([f'L={depth}\nedge', f'L={depth}\ncenter'])

    edge_vals = np.asarray(r['edge_mean_by_seed'])
    center_vals = np.asarray(r['center_mean_by_seed'])
    for edge_val, center_val in zip(edge_vals, center_vals):
        ax.plot([edge_x, center_x], [edge_val, center_val], color='0.75', linewidth=1.0, alpha=0.9)
    ax.scatter(np.full_like(edge_vals, edge_x, dtype=float), edge_vals, color='tab:red', s=20, alpha=0.8)
    ax.scatter(np.full_like(center_vals, center_x, dtype=float), center_vals, color='tab:blue', s=20, alpha=0.8)

    edge_mean = np.mean(edge_vals)
    center_mean = np.mean(center_vals)
    edge_ci = exp.ci95_half_width(edge_vals)
    center_ci = exp.ci95_half_width(center_vals)
    ax.errorbar([edge_x, center_x], [edge_mean, center_mean], yerr=[edge_ci, center_ci], fmt='o', color='k', capsize=4, linewidth=2)

ax.set_xticks(positions)
ax.set_xticklabels(labels)
ax.set_ylabel('mean error within summary region')
ax.set_title('Edge vs center summaries by seed')

# Right panel: edge-center contrast with correlation annotations
ax = axes[1]
x = np.arange(len(depths))
edge_center = [depth_results[d]['edge_minus_center_mean'] for d in depths]
edge_center_ci = [depth_results[d]['edge_minus_center_ci95'] for d in depths]
mean_corrs = [depth_results[d]['mean_profile_correlation'] for d in depths]
seed_corr_means = [depth_results[d]['per_seed_correlation_mean'] for d in depths]
seed_corr_ci = [depth_results[d]['per_seed_correlation_ci95'] for d in depths]

ax.bar(x, edge_center, yerr=edge_center_ci, color='tab:purple', alpha=0.75, capsize=5)
ax.axhline(0.0, color='black', linewidth=1)
for xi, depth, corr_mean, corr_seed, corr_ci in zip(x, depths, mean_corrs, seed_corr_means, seed_corr_ci):
    offset = 0.02 if edge_center[xi] >= 0 else -0.02
    ax.text(
        xi,
        edge_center[xi] + offset,
        f"mean corr={corr_mean:.2f}\nseed corr={corr_seed:.2f} ± {corr_ci:.2f}",
        ha='center',
        va='bottom' if edge_center[xi] >= 0 else 'top',
        fontsize=8,
    )
ax.set_xticks(x)
ax.set_xticklabels([f'L={depth}' for depth in depths])
ax.set_ylabel('edge mean - center mean')
ax.set_title('Boundary-distance contrast')

fig.subplots_adjust(bottom=0.20, top=0.88, wspace=0.32)
show_figure(fig)
plt.close(fig)


In [ ]:
verdict_rows = []
for depth in depths:
    r = depth_results[depth]
    edge_gt_center_count = int(np.sum(np.asarray(r['edge_minus_center_by_seed']) > 0.0))
    neg_corr_count = int(np.sum(np.asarray(r['per_seed_correlations']) < 0.0))
    verdict_rows.append([
        depth,
        f"{r['edge_minus_center_mean']:.4f} ± {r['edge_minus_center_ci95']:.4f}",
        f"{r['per_seed_correlation_mean']:.3f} ± {r['per_seed_correlation_ci95']:.3f}",
        f"{edge_gt_center_count}/{used_config['num_seeds']}",
        f"{neg_corr_count}/{used_config['num_seeds']}",
        'PASS' if r['tests']['legacy_T1_edge_layers_smallest'] else 'FAIL',
        'PASS' if r['tests']['legacy_T2_middle_peak'] else 'FAIL',
        'YES' if r['tests']['observed_edge_higher_than_center'] else 'NO',
        'YES' if r['tests']['observed_negative_distance_correlation'] else 'NO',
        f"{r['pathA_mean_final_loss']:.4e}",
        f"{r['pathB_mean_final_loss']:.4e}",
        f"{r['pathB_over_pathA_loss_ratio_mean']:.3f} ± {r['pathB_over_pathA_loss_ratio_ci95']:.3f}",
    ])

print('Current verdict table')
print(ascii_table(
    [
        'depth',
        'edge-center mean ±95% CI',
        'seed corr mean ±95% CI',
        'edge>center seeds',
        'corr<0 seeds',
        'legacy T1',
        'legacy T2',
        'edge>center?',
        'corr<0?',
        'mean Path A loss',
        'mean Path B loss',
        'B/A loss ratio',
    ],
    verdict_rows,
))

print('\nReadout guide:')
print('- "legacy T1" and "legacy T2" are retained to make the earlier framing auditable rather than silently discarded.')
print('- "edge>center?" and "corr<0?" summarize the actually observed boundary-distance pattern in the present run.')
print('- Compare Path A and Path B losses as a stability diagnostic: finite, comparable losses help show that the error metric is not merely tracking numerical blow-up.')
if SMOKE_MODE:
    print('- Because this is smoke mode, use the table only to confirm the pipeline runs end-to-end.')


---
## Calibrated Conclusion

The conclusion below is intentionally tied to the implemented metric and the observed output of
this exact toy study. It avoids stronger mechanistic claims than the experiment itself supports.


In [ ]:
legacy_t1_supported_depths = sum(1 for depth in depths if depth_results[depth]['tests']['legacy_T1_edge_layers_smallest'])
legacy_t2_supported_depths = sum(1 for depth in depths if depth_results[depth]['tests']['legacy_T2_middle_peak'])
all_edge_high = all_edge_higher_than_center(depth_results)
all_neg_corr = all_negative_mean_profile_correlations(depth_results)

print('Conclusion')
print('----------')
for depth in depths:
    r = depth_results[depth]
    center_layers = ', '.join(str(layer) for layer in r['center_layers'])
    print(
        f"Depth {depth}: edge mean error = {r['edge_mean_error']:.4f}, "
        f"center mean error (layers {center_layers}) = {r['center_mean_error']:.4f}, "
        f"corr(dist_from_edge, mean profile) = {r['mean_profile_correlation']:.3f}."
    )

print('\nTakeaway:')
if SMOKE_MODE:
    print('- This run used smoke-mode settings, so its purpose is execution validation rather than substantive scientific interpretation.')
    print('- For the actual toy-study conclusion, rerun the notebook with the default configuration.')
else:
    if all_edge_high and all_neg_corr:
        print('- In the default toy Muon/deep-linear setup, the measured target-layer equivariance error is edge-high / center-low rather than middle-peaked.')
        print('- Equivalently, the boundary-distance correlation of the mean profile is negative for every tested depth in this run.')
    else:
        print('- This run does not support a single uniform qualitative pattern across all tested depths; interpret the tables and profiles directly.')
    print(f'- Legacy T1 is supported on {legacy_t1_supported_depths}/{len(depths)} tested depths.')
    print(f'- Legacy T2 is supported on {legacy_t2_supported_depths}/{len(depths)} tested depths.')
    print('- Therefore the older edge-smallest / middle-largest narrative should not be treated as established by this experiment alone.')
    print('- The result remains narrow: it is evidence about final target-layer self-error after a single-layer conjugation, not a direct proof of cross-layer propagation or its mechanism.')
    print('- If the scientific target is propagation, the natural next experiment is to score all layers and/or full training trajectories after perturbing one layer.')
